In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random
from sqlalchemy import create_engine, text
import os

In [2]:
# conn = sqlite3.connect("./my_database.db")

In [3]:
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "vendor_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
conn = create_engine(DATABASE_URL)

In [4]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

OperationalError: (psycopg2.OperationalError) connection to server at "127.0.0.1", port 5432 failed: FATAL:  database "vendor_db" does not exist

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
vendor_ids

In [ ]:
test_vendor = vendor_ids.iloc[1]['vendor_id']

In [ ]:
test_vendor

In [ ]:
def active_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and status = 'IN_PROGRESS'
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [ ]:
active_tickets(test_vendor)

In [ ]:
def vendor_on_time_delivery(vendor_id):
    df = pd.read_sql(
        f"""
        select * 
        from ticket
        where vendor_id = '{vendor_id}'
        """,
        con=conn
    )
    df['on_time'] = (
        df['completed_at'].notna() &
        (df['completed_at'] <= df['due_date'])
    )

    df = df.groupby('vendor_id', as_index=False).agg(on_time_count=("on_time", "sum"),ticket_count=("ticket_id", "count"))
    df['on_time_pct'] = df['on_time_count'] / df['ticket_count'] * 100
    print(df[['vendor_id', 'on_time_count', 'ticket_count', 'on_time_pct']])
    on_time_pct = df['on_time_pct'].iloc[0].round(2)
    return {'vendor_id': vendor_id, 'on_time_pct': on_time_pct}

In [ ]:
vendor_on_time_delivery(test_vendor)

In [ ]:
def flagged_tickets(vendor_id):
    df = pd.read_sql(
        f"""
            select count(*) as flagged_tickets from ticket
            where vendor_id='{vendor_id}' and anomaly_flag=True
        """, con=conn
    )
    flagged_ticket_count = df['flagged_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'flagged_ticket_count': flagged_ticket_count}

In [ ]:
flagged_tickets(test_vendor)

## Completed Today (work in progress due to logic of sample data)

In [ ]:
df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{test_vendor}' and status='COMPLETED'
        """, parse_dates=['assigned_at', 'completed_at', 'created_at', 'updated_at', 'start_time', 'due_date'], con=conn
    )

In [ ]:
max(df['completed_at'])

In [ ]:
def completed_tickets_today(vendor_id):
    df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{vendor_id}'
        """, con=conn
    )
    return df

In [ ]:
# completed_tickets_today(test_vendor)

## Average Duration

In [ ]:
def average_duration(vendor_id):
    df = pd.read_sql(
        f"""
        select * from ticket
        where vendor_id = '{vendor_id}' and status='COMPLETED'
        """,
        con=conn,
        parse_dates=['completed_at', 'start_time']
    )
    df['duration'] = df['completed_at'] - df['start_time']
    avg_duration = df['duration'].mean()
    return {'vendor_id': vendor_id, 'avg_duration': avg_duration}

In [ ]:
average_duration(test_vendor)